# QueryBridge evaluation

Shows the controlled multi-query development result and the supporting-data lexicon size without using relevance labels during generation.

**Status:** development evidence; zero locked test queries used.


In [1]:
from pathlib import Path
import csv, hashlib, json, random, statistics
from collections import Counter
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SEED = 20250816
random.seed(SEED)
print(f"Project: {ROOT.name} | fixed seed: {SEED}")


Project: RAABTA_PROJECT_PORTABLE | fixed seed: 20250816


In [2]:
report = json.loads((ROOT / "reports/tables/querybridge_development.json").read_text(encoding="utf-8"))
lexicon = json.loads((ROOT / "artifacts/metadata/transliteration_lexicon.json").read_text(encoding="utf-8"))
assert report["test_queries_used"] == 0
print({"lexicon_entries": len(lexicon["entries"]), **report["result"]})


{'lexicon_entries': 10530, 'mean_accepted_variants': 3.467, 'mean_latency_ms': 444.043, 'mrr_at_10': 0.074987, 'ndcg_at_10': 0.095905, 'p95_latency_ms': 656.212, 'recall_at_1': 0.05, 'recall_at_10': 0.166667, 'recall_at_5': 0.1}


## Improvement over baselines


In [3]:
baseline = json.loads((ROOT / 'reports/tables/baselines_development.json').read_text(encoding='utf-8'))
for name, values in baseline['systems'].items():
    print(name, {metric: round(report['result'][metric] - values[metric], 6) for metric in ('recall_at_1', 'recall_at_5', 'recall_at_10', 'mrr_at_10', 'ndcg_at_10')})


direct_dense {'recall_at_1': 0.0, 'recall_at_5': 0.016667, 'recall_at_10': 0.075, 'mrr_at_10': 0.013459, 'ndcg_at_10': 0.027259}
single_transliteration_bm25 {'recall_at_1': 0.05, 'recall_at_5': 0.083333, 'recall_at_10': 0.141667, 'mrr_at_10': 0.067547, 'ndcg_at_10': 0.084281}
standard_hybrid {'recall_at_1': 0.025, 'recall_at_5': 0.033333, 'recall_at_10': 0.075, 'mrr_at_10': 0.029385, 'ndcg_at_10': 0.039391}


## Lexicon coverage


In [4]:
lexicon = json.loads((ROOT / 'artifacts/metadata/transliteration_lexicon.json').read_text(encoding='utf-8'))['entries']
with (ROOT / 'data/diagnostic/raabta_diagnostic.csv').open(encoding='utf-8-sig', newline='') as handle:
    development = [row for row in csv.DictReader(handle) if row['split'] == 'development']
import re
coverage = []
for row in development:
    tokens = re.findall(r'[a-z0-9]+', row['roman_urdu_query'].lower())
    coverage.append(sum(token in lexicon for token in tokens) / len(tokens) if tokens else 0)
print({'lexicon_entries': len(lexicon), 'mean_token_coverage': round(statistics.fmean(coverage), 4), 'full_coverage_queries': sum(value == 1 for value in coverage)})


{'lexicon_entries': 10530, 'mean_token_coverage': 0.3191, 'full_coverage_queries': 1}


## Interpretation

Outputs above are measurements from frozen local artifacts. Important limitations must remain attached when reused in the report or viva.
